In [1]:
%pip install "numpy" "opencv-python" -q
%pip install git+https://www.github.com/mouseland/cellpose.git
%pip install pyocclient -q

  Cloning https://www.github.com/mouseland/cellpose.git to /tmp/pip-req-build-w857m6rq
  Running command git clone --filter=blob:none --quiet https://www.github.com/mouseland/cellpose.git /tmp/pip-req-build-w857m6rq
  Resolved https://www.github.com/mouseland/cellpose.git to commit a9f8bfcde43033247309e3982747df9fe9f09315
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 59.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 77.6 MB/s eta 0:00:00:00:0100:01
  Created wheel for cellpose: filename=cellpose-4.1.1-py3-none-any.whl size=213288 sha256=d5aefe2333fcfb275a7fde607323b8b0eadb58ef6bbb7a1d9d4b5f17da4e4c09
  Stored in directory: /tmp/pip-ephem-wheel-cache-cuj90onk/wheels/df/b6/31/a3013c44290eabb46f4c06d1efb19744124fcad2d59684ec5e
Successfully built cellpose
  Preparing metadata (setup.py) ... done


In [1]:
import owncloud, getpass, cellpose

# Config
url, user = 'https://cloud.minesparis.psl.eu', 'gabriel.gautier'
oc_session = owncloud.Client(url)
oc_session.login(user, getpass.getpass(f"PW {user}: "))
oc_session.get_file('/travail/Mines/DIMA/Segmentation/scripts/tool.py', 'tool.py')

import tool
oc = tool.Owncloud(oc_session)

print("Tool chargé et prêt.")

Tool chargé et prêt.


In [24]:
oc.upload_data()

In [13]:
oc.upload_scripts()

In [10]:
import os, shutil
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tool
import pandas as pd


from cellpose import core, utils, io, models, metrics, train, dynamics, transforms, plot
from glob import glob

import importlib
importlib.reload(tool)
import tool

visualizer = tool.CellVisualizer()

In [6]:

oc.upload_models()

In [25]:
import json

with open('./data/dataset_SB_splits.json', 'r') as f:
    splits = json.load(f)

ids_test  = splits['test']

print(f"Splits chargés : Test({len(ids_test)})")

Splits chargés : Test(15)


In [4]:
df_annotations = pd.read_csv('./data/Nuclei.csv')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from cellpose import train, models, io

label = "Nuclei_cellpose4"
target = "Nuc"                                    
initial_model_path = "./models/cellpose4_nuclei.pth" 
z_levels = [0, 1, 2, 3]                          

# Paramètres d'inférence
eval_params = {
    "channels": [0, 0],
    "diameter": None,
    "flow_threshold": 0.4,
    "cellprob_threshold": -0.5
}

# Paramètres d'entraînement
train_params = {
    "n_epochs": 40,
    "learning_rate": 1e-5,
    "weight_decay": 0.0001,
    "save_path": '.'
}

io.logger_setup()
current_model_path = initial_model_path

for i in range(len(z_levels) - 1):
    z_current = z_levels[i]
    z_next = z_levels[i+1]

    print(f"\n{'='*60}")
    print(f"ÉTAPE {i+1} : Apprentissage du plan Z={z_next} à partir de Z={z_current}")
    print(f"{'='*60}")

    # --- A. Chargement des images cibles (Z+1) ---
    X_train_next = visualizer.load_set_z(ids_train, mode=target, z_level=z_next)
    X_val_next = visualizer.load_set_z(ids_val, mode=target, z_level=z_next)

    # --- B. Génération des pseudo-labels ---
    print(f"Génération des pseudo-labels pour Z={z_next} avec {label}...")
    model_eval = models.CellposeModel(gpu=True, pretrained_model=current_model_path)
    
    # L'opérateur ** déballe le dictionnaire directement dans les arguments de la fonction
    Y_train_pseudo, _, _, _ = model_eval.eval(X_train_next, **eval_params)
    Y_val_pseudo, _, _, _ = model_eval.eval(X_val_next, **eval_params)

    print(f"finetuning du modèle pour Z={z_next}...")
    model_finetune = models.CellposeModel(gpu=True, pretrained_model=current_model_path)
    
    model_name = f"{label}_z{z_next}"

    new_model_path, t_loss, v_loss = train.train_seg(
        model_finetune.net,
        train_data=X_train_next,
        train_labels=Y_train_pseudo,
        test_data=X_val_next,
        test_labels=Y_val_pseudo,
        model_name=model_name,
        **train_params 
    )

    print(f"Modèle Z={z_next} sauvegardé : {new_model_path}")

    title = f"Loss ({label} Z={z_next}) LR={train_params['learning_rate']}"
    visualizer.plot_loss(t_loss, v_loss, title=title, show=True)

    current_model_path = new_model_path

print("\nEntraînement séquentiel terminé avec succès.")

In [ ]:
oc.oc.put_file(
    '/travail/Mines/DIMA/Segmentation/results/models/cellpose4_nuclei_stack.pth',
    'models/cellpose4_nuclei_stack', timeout=600                                  
)